# Projet : Composantes connexes avec Spark RDD

## Implémentation de l'algorithme CCF (Connected Component Finder)

Ce notebook présente une implémentation complète de l'algorithme CCF en utilisant exclusivement l'API RDD de Spark.
L'objectif est de calculer les composantes connexes d'un graphe non orienté via une propagation itérative du plus petit identifiant.


## Initialisation de Spark

In [ ]:

try:
    spark
except NameError:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.appName("CCF_RDD_Project").getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("WARN")
print("Version Spark :", spark.version)


Version Spark : 4.0.2


## Fonctions utilitaires

In [ ]:

def rendre_non_oriente(edges):
    # Pour chaque arête (u,v) on ajoute (v,u)
    return edges.flatMap(lambda e: [e, (e[1], e[0])])

def dedup(pairs):
    # Suppression des doublons (phase CCF-Dedup)
    return pairs.distinct()


## Version 1 : Implémentation basique (groupByKey)

In [ ]:

def ccf_iterate_basique(pairs):

    grouped = pairs.groupByKey()

    def propagation(kv):
        u, voisins_iter = kv
        voisins = list(voisins_iter)

        if not voisins:
            return []

        # Minimum entre le sommet et ses voisins
        m = min([u] + voisins)

        # Si déjà minimum, on ne fait rien
        if m >= u:
            return []

        sorties = [(u, m)]

        # Propagation vers les voisins
        for v in voisins:
            if v != m:
                sorties.append((v, m))

        return sorties

    return grouped.flatMap(propagation)


## Version 2 : Implémentation améliorée (reduceByKey + join)

In [ ]:

def ccf_iterate_join(pairs):

    # Ajout (u,u) pour comparaison interne
    self_pairs = pairs.map(lambda x: (x[0], x[0]))

    union_pairs = pairs.union(self_pairs)

    # Calcul du minimum par sommet
    min_id = union_pairs.reduceByKey(lambda a,b: min(a,b))

    # Jointure pour propager aux voisins
    joined = pairs.join(min_id)

    prop_v = joined.map(lambda x: (x[1][0], x[1][1]))

    prop_u = min_id.filter(lambda x: x[1] < x[0])

    return prop_v.union(prop_u)


## Boucle principale

In [ ]:

def run_ccf(edges, iterate_fn, max_iter=30):

    pairs = rendre_non_oriente(edges).distinct()

    for i in range(max_iter):

        new_pairs = iterate_fn(pairs)
        new_pairs = dedup(new_pairs)

        diff = new_pairs.subtract(pairs)
        nb_new = diff.count()

        print("Iteration", i+1, "- nouvelles paires :", nb_new)

        if nb_new == 0:
            pairs = new_pairs
            break

        pairs = new_pairs

    composantes = pairs.reduceByKey(lambda a,b: min(a,b))
    return composantes


## Test simple

In [ ]:

edges = sc.parallelize([(1,2),(2,3),(10,11)])
res = run_ccf(edges, ccf_iterate_basique)
print("Résultat :", sorted(res.collect()))


Iteration 1 - nouvelles paires : 1
Iteration 2 - nouvelles paires : 0
Résultat : [(2, 1), (3, 1), (11, 10)]


## Analyse expérimentale simple

In [ ]:

from time import perf_counter

def graphe_chaine(n):
    return [(i,i+1) for i in range(n-1)]

def benchmark(n):
    print("\nTaille :", n)
    edges = sc.parallelize(graphe_chaine(n))
    start = perf_counter()
    run_ccf(edges, ccf_iterate_join)
    print("Temps :", perf_counter()-start)

for taille in [500, 2000, 5000]:
    benchmark(taille)



Taille : 500
Iteration 1 - nouvelles paires : 997
Iteration 2 - nouvelles paires : 0
Temps : 57.940157639999995

Taille : 2000
Iteration 1 - nouvelles paires : 3997
Iteration 2 - nouvelles paires : 0
Temps : 54.02111997999998

Taille : 5000
Iteration 1 - nouvelles paires : 9997
Iteration 2 - nouvelles paires : 0
Temps : 55.49622146500002


## Conclusion
L'algorithme CCF fonctionne par propagation itérative du minimum.
La version utilisant reduceByKey + join est généralement plus performante.
